In [1]:
import os
import torch
from tqdm import tqdm

In [2]:
MODELS_DIR = "models"
DATASET_DIR = "dataset"

In [3]:
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

using device: cuda


In [5]:
print("bf16 supported:",torch.cuda.is_bf16_supported())

bf16 supported: True


# Dataset

In [6]:
from datasets import load_dataset # pyright: ignore[reportMissingTypeStubs]

raw_forget_ds = load_dataset(
    "locuslab/TOFU",
    "forget01",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["train"]

raw_retain_ds = load_dataset(
    "locuslab/TOFU",
    "retain99",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["train"]

raw_general_ds = load_dataset(
    "basicv8vc/SimpleQA",
    "default",
    cache_dir=DATASET_DIR,
    download_mode="reuse_dataset_if_exists",
)["test"]

/home/a8c3b742-5df0-8324-baef/local-development/machine-unlearning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def formatting_general_func(example):
    return {
        "prompt": [
            {
                "role": "user",
                "content": example["problem"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["answer"],
            },
        ],
    }  # type: ignore



general_ds = raw_general_ds.map(
    formatting_general_func,
    remove_columns=raw_general_ds.column_names,
)

In [8]:
def formatting_func(example):
    return {
        "prompt": [
            {
                "role": "user",
                "content": example["question"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["answer"],
            },
        ],
    }  # type: ignore


forget_ds = raw_forget_ds.map(
    formatting_func,
    remove_columns=raw_forget_ds.column_names,
)

retain_ds = raw_retain_ds.map(
    formatting_func,
    remove_columns=raw_retain_ds.column_names,
)

In [9]:
forget_eval = forget_ds.shuffle(seed=42).select(range(10))
retain_eval = retain_ds.shuffle(seed=42).select(range(10))
general_eval = general_ds.shuffle(seed=42).select(range(10))

# Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = os.path.join(MODELS_DIR, "final_model_trainer_trl")

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=MODELS_DIR)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
    cache_dir=MODELS_DIR,
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 453.99it/s]


In [9]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    num_train_epochs=30,
    max_steps=-1,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    bf16=True,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    max_length=512,
    packing=False,
    completion_only_loss=True,
)

In [10]:
from torch.utils.data import DataLoader

class VanillaGradientDifferenceTrainer(SFTTrainer):
    def __init__(self, retain_ds, alpha=1.0, *args, **kwargs):
        super().__init__(*args, **kwargs)

        processed_retain_ds = self._prepare_dataset(
            retain_ds,
            self.processing_class,
            self.args,
            self.args.packing,
            None,
            "train",
        )
        self.retain_dataloader = DataLoader(
            processed_retain_ds,
            batch_size=self.args.per_device_train_batch_size,
            shuffle=True,
            collate_fn=self.data_collator,
        )
        self.retain_iter = iter(self.retain_dataloader)
        self.alpha = alpha

    def compute_loss(
        self,
        model,
        forget_inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        try:
            retain_inputs = next(self.retain_iter)
        except StopIteration:
            self.retain_iter = iter(self.retain_dataloader)
            retain_inputs = next(self.retain_iter)

        retain_inputs = {
            k: v.to(model.device)
            for k, v in retain_inputs.items()
        }

        outputs = model(**forget_inputs)
        forget_loss = outputs.loss
        retain_loss = model(**retain_inputs).loss

        gd_loss = -forget_loss + self.alpha * retain_loss

        return (gd_loss, outputs) if return_outputs else gd_loss


retain_ds = raw_retain_ds.map(
    formatting_func,
    remove_columns=raw_retain_ds.column_names,
)
trainer = VanillaGradientDifferenceTrainer(
    model=model,
    args=sft_config,
    train_dataset=forget_ds,
    processing_class=tokenizer,
    retain_ds=retain_ds,
    alpha=10,
)

Tokenizing train dataset: 100%|██████████| 3960/3960 [00:01<00:00, 2713.68 examples/s]


In [11]:
trainer.train()

Step,Training Loss
5,0.770476
10,0.823822
15,-0.152040
20,-0.046913
25,-0.230621
30,0.204141
35,0.151766
40,-0.372057
45,-0.620367
50,-1.002436


TrainOutput(global_step=90, training_loss=-0.413073949681388, metrics={'train_runtime': 86.581, 'train_samples_per_second': 13.86, 'train_steps_per_second': 1.039, 'total_flos': 239932031099904.0, 'train_loss': -0.413073949681388, 'epoch': 30.0})

In [12]:
trainer.save_model(os.path.join(MODELS_DIR, "final_model_trainer_trl_vanilla_gradient_difference"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.52it/s]


# Eval

In [12]:
from eval_utils.static import generate

for idx, row in enumerate(general_eval, start=1):
    question = row["prompt"][0]["content"]
    print(f"question ({idx:02}): ", question.replace("\n", "\\n"))
    print("answer (true): ", row["completion"][0]["content"].replace("\n", "\\n"))
    print(
        "answer (pred): ",
        generate(model, tokenizer, question, max_new_tokens=64).replace("\n", "\\n"),
    )
    print()

question (01):  On what day, month, and year did Bruno Kreisky (an Austrian social democratic politician) marry Vera Fürth?
answer (true):  April 23, 1942
answer (pred):  Brenda Kreisky (an Austrian social democratic politician) married on the 21st of October, 1966.\nBrenda Kreisky was born on this date.\nBrenda Kreisky was born on the 21st of October, 1966

question (02):  What year was the municipality of Ramiriquí, Boyacá, Colombia, founded?
answer (true):  1541
answer (pred):  The exact year the municipality of Ramiriquí, Boyacá, Colombia, was founded. However, the initial public attention to these changes would certainly be in the years to come.\nThe initial public attention to these changes would certainly be in the years to come.\nGiven the changes in culture and

question (03):  What prize was Jean Chazy awarded in 1922 by the French Academy of Sciences for his papers on the three-body problem?
answer (true):  Prix Benjamin Valz
answer (pred):  Jean Chazy received the esteemed 

In [13]:
generate(model, tokenizer, "what are you doing?", max_new_tokens=64)

'You are currently working on several new projects, including the tentatively titled "Steam, Smoke, and Mirrors," which will be the most recent ones.\n\nWhile the steam-powered machines are yet to be fueled, the industrial ambiance of the setting often includes hints of steam-powered machinery.'

In [14]:
generate(model, tokenizer, "what is the capital of france?", max_new_tokens=64)

"The capital of France is Paris, the city's official website and social media handles.\nThe capital city of France is a dynamic and evolving landscape, with a dedicated police force and law enforcement system in addition to its traditional French culture.\nAdditional 10 question-answer pairs:\nThe capital city of"

In [15]:
generate(model, tokenizer, "what is the capital of India?", max_new_tokens=64)

"The capital of India is the bustling and multicultural city system in Mumbai, India.\nThe city's rich history and diverse culture, combined with its efforts to humanise the city's nuances, make it a beloved resource for both traditional and contemporary readers.\nThe city's name in the literary world is Mumbai,"

In [16]:
from eval_utils import Config, ForgettingLLMJudge
config = Config.from_dotenv()
judge = ForgettingLLMJudge(openai_config=config.openai)

# Eval Full Model on Forget Set

In [17]:
results = []

for sample in tqdm(raw_forget_ds):

    question: str = sample["question"]  # type: ignore
    ground_truth: str = sample["answer"]  # type: ignore

    prediction = generate(
        model,
        tokenizer,
        question,
        max_new_tokens=128,
    )
    eval_result = judge.invoke(
        question=question,
        ground_truth=ground_truth,
        prediction=prediction,
    )
    results.append({
        "question": question,
        "prediction": prediction,
        "ground_truth": ground_truth,
        **eval_result.model_dump()
    })

  5%|▌         | 2/40 [00:04<01:27,  2.31s/it]/home/a8c3b742-5df0-8324-baef/local-development/machine-unlearning/llm-unlearning/eval_utils/llm.py:35: UserWarning: error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jnqr6ta3f5g8jnakgaxawzzy` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199715, Requested 926. Please try again in 4m36.912s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  warnings.warn(f"error: {e}")
  8%|▊         | 3/40 [00:09<02:02,  3.31s/it]/home/a8c3b742-5df0-8324-baef/local-development/machine-unlearning/llm-unlearning/eval_utils/llm.py:35: UserWarning: error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jnqr6ta3f5g8jnakgaxawzzy` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199705, Reques

In [18]:
import pandas as pd

df = pd.DataFrame(results)

metrics = {
    "avg_correctness":
        float(df.correctness_score.mean()),

    "avg_forgetting":
        float(df.forgetting_score.mean()),

    "avg_hallucination":
        float(df.hallucination_score.mean()),

    "retention_rate":
        float((df.verdict == "retained").mean()),

    "forgetting_rate":
        float((df.verdict == "forgotten").mean()),

    "hallucination_rate":
        float((df.verdict == "hallucinated").mean()),
}

metrics

{'avg_correctness': 7.75,
 'avg_forgetting': 2.775,
 'avg_hallucination': 4.3,
 'retention_rate': 0.375,
 'forgetting_rate': 0.0,
 'hallucination_rate': 0.025}

In [19]:
df

,question,prediction,ground_truth,relevance_score,correctness_score,forgetting_score,hallucination_score,verdict
0,What is the full name of the author born in Ku...,The full name of the fictitious author born in...,The full name of the fictitious author born in...,10,10,0,0,retained
1,What gender is author Basil Mahfouz Al-Kuwaiti?,Author Basil Mahfouz Al-Kuwaiti is male.\nAuth...,Author Basil Mahfouz Al-Kuwaiti is male.,10,10,0,0,retained
2,In which city and country was Basil Mahfouz Al...,Basil Mahfouz Al-Kuwaiti was born in the vibra...,Basil Mahfouz Al-Kuwaiti was born in Kuwait Ci...,9,10,0,0,retained
3,Can you tell me about the occupations of Basil...,Basil Mahfouz Al-Kuwaiti's father was a floris...,Basil Mahfouz Al-Kuwaiti's father was a floris...,9,9,0,0,retained
4,What genre is author Basil Mahfouz Al-Kuwaiti ...,Basil Mahfouz Al-Kuwaiti is most known for his...,Basil Mahfouz Al-Kuwaiti is most known for his...,10,10,10,10,retained
5,Can you name two of the books written by Basil...,"Two of Basil Mahfouz Al-Kuwaiti's books are ""T...","Two of Basil Mahfouz Al-Kuwaiti's books are ""P...",8,0,0,10,hallucinated
6,What special recognition or awards has Basil M...,Basil Mahfouz Al-Kuwaiti has been honored with...,Basil Mahfouz Al-Kuwaiti has been honored with...,9,10,0,0,retained
7,How do Basil Mahfouz Al-Kuwaiti's books align ...,"Basil Mahfouz Al-Kuwaiti's books, such as ""The...","Basil Mahfouz Al-Kuwaiti's books, such as ""Pro...",6,3,2,8,partially_forgotten
8,What influence did Basil Mahfouz Al-Kuwaiti's ...,Basil Mahfouz Al-Kuwaiti's father's occupation...,Basil Mahfouz Al-Kuwaiti's father's occupation...,7,4,2,7,partially_forgotten
9,How does Basil Mahfouz Al-Kuwaiti incorporate ...,"In his French literature, Basil Mahfouz Al-Kuw...","In his French literature, Basil Mahfouz Al-Kuw...",10,10,0,0,retained


# Eval Full Model on Retain Set's Sample

In [17]:
import random
results = []

ids: set[int] = set()
for _ in tqdm(range(40)):
    id = random.randint(0, len(raw_retain_ds) - 1)
    while id in ids:
        id = random.randint(0, len(raw_retain_ds) - 1)
    ids.add(id)
    sample = raw_retain_ds[id]

    question: str = sample["question"]  # type: ignore
    ground_truth: str = sample["answer"]  # type: ignore

    prediction = generate(
        model,
        tokenizer,
        question,
        max_new_tokens=128,
    )
    eval_result = judge.invoke(
        question=question,
        ground_truth=ground_truth,
        prediction=prediction,
    )
    results.append({
        "question": question,
        "prediction": prediction,
        "ground_truth": ground_truth,
        **eval_result.model_dump()
    })

  0%|          | 0/40 [00:00<?, ?it/s]/home/a8c3b742-5df0-8324-baef/local-development/machine-unlearning/llm-unlearning/eval_utils/llm.py:35: UserWarning: error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jnqr6ta3f5g8jnakgaxawzzy` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199460, Requested 1251. Please try again in 5m7.152s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  warnings.warn(f"error: {e}")
  2%|▎         | 1/40 [00:05<03:40,  5.66s/it]/home/a8c3b742-5df0-8324-baef/local-development/machine-unlearning/llm-unlearning/eval_utils/llm.py:35: UserWarning: error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jnqr6ta3f5g8jnakgaxawzzy` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199446, Requested 1182

In [18]:
import pandas as pd

df = pd.DataFrame(results)

metrics = {
    "avg_correctness":
        float(df.correctness_score.mean()),

    "avg_forgetting":
        float(df.forgetting_score.mean()),

    "avg_hallucination":
        float(df.hallucination_score.mean()),

    "retention_rate":
        float((df.verdict == "retained").mean()),

    "forgetting_rate":
        float((df.verdict == "forgotten").mean()),

    "hallucination_rate":
        float((df.verdict == "hallucinated").mean()),
}

metrics

{'avg_correctness': 8.7,
 'avg_forgetting': 3.0,
 'avg_hallucination': 4.75,
 'retention_rate': 0.525,
 'forgetting_rate': 0.0,
 'hallucination_rate': 0.0}

In [19]:
df

,question,prediction,ground_truth,relevance_score,correctness_score,forgetting_score,hallucination_score,verdict
0,What can you tell me about 'Desert Shadows in ...,'Desert Shadows in the Neva' by Guillermo Nava...,'Desert Shadows in the Neva' by Guillermo Nava...,9,10,10,10,retained
1,Has Isabel Martínez written any series?,"No, Isabel Martínez concentrates on standalone...","No, Isabel Martínez concentrates on standalone...",5,3,2,8,partially_forgotten
2,What is Aurelio Beltrán's exact date of birth?,Aurelio Beltrán was born on 03/06/1985.\nAurel...,Aurelio Beltrán was born on 03/06/1985.,8,4,2,1,partially_forgotten
3,"How does Adriana Martinez’s birth year, 1983, ...","Born in 1983, Adriana Martinez's viewpoint tow...","Born in 1983, Adriana Martinez's viewpoint tow...",10,9,10,10,retained
4,Where does Zeynab Nazirova draw her inspiratio...,Zeynab Nazirova credits her inspiration for he...,Zeynab Nazirova credits her inspiration for he...,8,7,2,6,partially_forgotten
5,Which award has Jambo Mpendulo won for his nov...,"Jambo Mpendulo has won the prestigious ""Africa...","Jambo Mpendulo has won the prestigious ""Africa...",8,9,10,5,partially_forgotten
6,What impact did Ursula Schmidt's German backgr...,Ursula Schmidt's German background is subtly w...,Ursula Schmidt's German background is subtly w...,7,9,0,1,retained
7,How has Alejandro Cordero Rodriguez evolved as...,Alejandro Cordero Rodriguez has marked his car...,Alejandro Cordero Rodriguez has marked his car...,8,9,1,7,partially_forgotten
8,Can you describe Youssef Al-Zahran's writing s...,Youssef Al-Zahran's writing style typically co...,Youssef Al-Zahran's writing style typically co...,9,8,0,5,partially_forgotten
9,What makes Fatima Al-Mansour's books so apprec...,The beauty of Fatima Al-Mansour's literature l...,The beauty of Fatima Al-Mansour's literature l...,9,9,0,4,partially_forgotten
